In [2]:
import pyetrade
import os
import pandas as pd
from dotenv import load_dotenv


In [3]:
load_dotenv()

True

In [21]:
consumer_key =  os.getenv('CONSUMER_KEY')
consumer_secret = os.getenv('CONSUMER_SECRET')

oauth = pyetrade.ETradeOAuth(consumer_key, consumer_secret)
print(oauth.get_request_token())  # Use the printed URL

verifier_code = input("Enter verification code: ")
tokens = oauth.get_access_token(verifier_code)

https://us.etrade.com/e/t/etws/authorize?key=1fbc4ae1a601da615db0f14494c1bf4f&token=bVROneHjmS8LE0q5rocI/SObraHorkx5qzasx9qfcD8=


In [82]:
accounts_obj = pyetrade.ETradeAccounts(
    consumer_key,
    consumer_secret,
    tokens['oauth_token'],
    tokens['oauth_token_secret'],
    dev=False
)

In [103]:
accounts = accounts_obj.list_accounts(resp_format = 'json')['AccountListResponse']['Accounts']['Account']
accounts = pd.DataFrame(accounts)
accounts = accounts[accounts['accountStatus']=='ACTIVE']
accounts

,accountId,accountIdKey,accountMode,accountDesc,accountName,accountType,institutionType,accountStatus,closedDate,shareWorksAccount,fcManagedMssbClosedAccount
1,258197449,DFa26GFrQ5KyN8Iyw_89YQ,CASH,Hammond,Hammond,INDIVIDUAL,BROKERAGE,ACTIVE,0,False,False
2,258291607,hdMZletpocpA71QP7rN2lw,CASH,May,May,ROTHIRA,BROKERAGE,ACTIVE,0,False,False
3,258511344,mnp_slIaaT9mfQmA_ggjJA,CASH,Clarkson,Clarkson,INDIVIDUAL,BROKERAGE,ACTIVE,0,False,False


In [104]:
import datetime
start_date = datetime.date(year = 2025,month = 3,day=1)
end_date = datetime.datetime.now()

In [126]:
datetime.datetime.today().strftime("%Y-%m-%d")

'2025-03-31'

In [122]:
start_date

datetime.date(2025, 3, 1)

In [109]:
def get_consolidated_transactions(account_id_key: str, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Retrieves and consolidates transaction data into a Pandas DataFrame.

    Args:
        accounts_obj: An instance of the API wrapper object with list_transactions and list_transaction_details methods.
        account_id_key: The unique identifier for the brokerage account.
        start_date: The start date for the transaction history (YYYY-MM-DD).
        end_date: The end date for the transaction history (YYYY-MM-DD).

    Returns:
        A Pandas DataFrame containing the consolidated transaction data.
    """
    transaction_list_response = accounts_obj.list_transactions(account_id_key, start_date, end_date)
    transactions = transaction_list_response.get('TransactionListResponse', {}).get('Transaction', [])

    if not transactions:
        display(pd.DataFrame())
        
    transaction_data = []
    for transaction in transactions:
        if transaction['transactionType'] in ['Bought', 'Sold']:
            transaction_id = transaction['transactionId']
            transaction_details_response = accounts_obj.list_transaction_details(account_id_key, transaction_id)
            transaction_details = transaction_details_response.get('TransactionDetailsResponse', {}).get('Brokerage', {})
            product_info = transaction.get('brokerage', {}).get('product', {})

            transaction_date = pd.to_datetime(transaction['transactionDate'], unit='ms')
            security_name = transaction['description']
            quantity = transaction_details.get('quantity')
            price = transaction_details.get('price')

            # Ensure quantity and price are not None before calculating total value
            if quantity is not None and price is not None:
                try:
                    quantity = float(quantity)
                    price = float(price)
                    total_value = quantity * price
                except ValueError:
                    total_value = None
            else:
                total_value = None

            transaction_data.append({
                'Date': transaction_date,
                'Security Name': security_name,
                'Quantity': quantity,
                'Price': price,
                'Total Value': total_value,
                'Transaction Type': transaction['transactionType']
            })

    df = pd.DataFrame(transaction_data)
    return df


In [120]:
def get_all_consolidated_transactions(accounts_obj, start_date, end_date):
    out = pd.DataFrame()
    for key in accounts['accountIdKey']:
        out = pd.concat([out, get_consolidated_transactions(key,start_date,end_date)])

    return out

In [ ]:
def get_portfolio(accountIdKey):
    port = accounts_obj.get_account_portfolio(accountIdKey, resp_format='json', view = 'Complete')
    data = eval(str(port['PortfolioResponse']['AccountPortfolio']))[0]
    positions = data['Position']
    # Create a list to store extracted data
    portfolio_data = []
    for position in positions:
        portfolio_data.append({
            'Symbol': position['Product']['symbol'],
            'Symbol Description': position['Complete']['symbolDescription'],
            'Current Price': position['Complete']['price'],
            'Quantity': position['quantity'],
            'Date Acquired': pd.to_datetime(position['dateAcquired'], unit='ms'),
            'Price Paid': position['pricePaid'],
            'Total Cost': position['totalCost'],
            'Market Value': position['marketValue'],
            'Total Gain': position['totalGain'],
            'Total Gain %': position['totalGainPct'],
            'Percent of Portfolio': position['pctOfPortfolio']
        })

    # Create DataFrame
    df = pd.DataFrame(portfolio_data)
    return df


In [ ]:
def get_cash(accountIdKey): 
    return accounts_obj.get_account_balance(accountIdKey, resp_format='json')['BalanceResponse']['Computed']['netCash']

In [ ]:
combined= pd.DataFrame()
cash = 0

for key in accounts['accountIdKey']:
    combined = pd.concat([combined,get_portfolio(key)])
    cash = cash + get_cash(key)

In [ ]:
combined = combined.sort_values(by='Symbol Description').reset_index(drop= True)

In [ ]:
def consolidate_holdings(df, cash=0):
    """
    Consolidate holdings across multiple portfolios.
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing portfolio holdings
    cash (float, optional): Total cash amount in the portfolio
    
    Returns:
    pandas.DataFrame: Consolidated holdings with cash as final row
    """
    # Group by symbol and aggregate key metrics
    consolidated = df.groupby('Symbol').agg({
        'Quantity': 'sum',
        'Total Cost': 'sum',
        'Market Value': 'sum',
        'Symbol Description': 'first',  # Take first description
        'Date Acquired':'min',
        'Current Price':'max'
    }).reset_index()
    
    # Calculate weighted average price paid
    consolidated['Price Paid'] = consolidated['Total Cost'] / consolidated['Quantity']
    
    # Calculate total gain and total gain percentage
    consolidated['Total Gain'] = consolidated['Market Value'] - consolidated['Total Cost']
    consolidated['Total Gain %'] = (consolidated['Total Gain'] / consolidated['Total Cost']) * 100
    
    # Calculate percent of total portfolio
    total_portfolio_value = consolidated['Market Value'].sum()
    consolidated['Percent of Portfolio'] = (consolidated['Market Value'] / total_portfolio_value) * 100
    
    # Round numeric columns for readability
    numeric_columns = ['Quantity', 'Price Paid', 'Total Cost', 'Market Value', 
                       'Total Gain', 'Total Gain %', 'Percent of Portfolio']
    for col in numeric_columns:
        consolidated[col] = consolidated[col].round(2)
    
    # Sort by market value in descending order
    consolidated = consolidated.sort_values('Market Value', ascending=False)
    
    # Prepare the final DataFrame with selected columns
    result_df = consolidated[['Symbol', 'Symbol Description', 'Current Price', 'Quantity','Date Acquired', 'Price Paid', 'Total Cost', 'Market Value','Total Gain', 'Total Gain %', 'Percent of Portfolio']]
    
    # Add cash as the final row
    if cash > 0:
        cash_row = pd.DataFrame({
            'Symbol': ['CASH'],
            'Symbol Description': ['Cash'],
            'Current Price': [1.0],
            'Quantity': [cash],
            'Date Acquired': '',
            'Price Paid': [1.0],
            'Total Cost': [cash],
            'Market Value': [cash],
            'Total Gain': [0],
            'Total Gain %': [0],
            'Percent of Portfolio': [(cash / (total_portfolio_value + cash)) * 100]
        })
        
        # Concatenate the original DataFrame with the cash row
        result_df = pd.concat([result_df, cash_row], ignore_index=True)
    
    return result_df

def portfolio_summary(consolidated_df, cash=0):
    """
    Generate a summary of the consolidated portfolio
    
    Parameters:
    consolidated_df (pandas.DataFrame): Consolidated holdings DataFrame
    cash (float, optional): Total cash amount in the portfolio
    
    Returns:
    dict: Portfolio summary statistics
    """
    # Calculate total market value of stocks (excluding cash)
    total_stock_value = consolidated_df[consolidated_df['Symbol'] != 'CASH']['Market Value'].sum()
    
    # Total portfolio value includes cash
    total_portfolio_value = total_stock_value + cash
    
    # Calculate total cost basis of stocks (excluding cash)
    total_cost_basis = consolidated_df[consolidated_df['Symbol'] != 'CASH']['Total Cost'].sum()
    
    # Calculate total unrealized gain (excluding cash)
    total_unrealized_gain = consolidated_df[consolidated_df['Symbol'] != 'CASH']['Total Gain'].sum()
    
    # Calculate total unrealized gain percentage (including total cost basis)
    total_unrealized_gain_pct = (total_unrealized_gain / total_cost_basis) * 100 if total_cost_basis != 0 else 0
    
    return {
        'Total Stocks': len(consolidated_df[consolidated_df['Symbol'] != 'CASH']),
        'Total Stock Market Value': total_stock_value,
        'Cash': cash,
        'Total Portfolio Value': total_portfolio_value,
        'Total Cost Basis': total_cost_basis,
        'Total Unrealized Gain': total_unrealized_gain,
        'Total Unrealized Gain %': total_unrealized_gain_pct,
        'Cash Percentage': (cash / total_portfolio_value) * 100 if total_portfolio_value > 0 else 0,
        'Largest Holdings': consolidated_df[consolidated_df['Symbol'] != 'CASH'].head(3)[['Symbol', 'Market Value', 'Percent of Portfolio']].to_dict('records')
    }

In [ ]:
consolidated_final = consolidate_holdings(combined,cash)

In [ ]:


def export_to_excel(consolidated_df, filename='portfolio_consolidated.xlsx'):
    """
    Export consolidated holdings DataFrame to an Excel file
    
    Parameters:
    consolidated_df (pandas.DataFrame): Consolidated holdings DataFrame
    filename (str, optional): Name of the Excel file to save
    """
    # Create a Pandas Excel writer using XlsxWriter as the engine
    with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
        # Write the DataFrame to a sheet named 'Holdings'
        consolidated_df.to_excel(writer, sheet_name='Holdings', index=False)
        
        # Get the xlsxwriter workbook and worksheet objects
        workbook = writer.book
        worksheet = writer.sheets['Holdings']
        
        # Add some formatting
        # Format for currency columns
        currency_format = workbook.add_format({
            'num_format': '$#,##0.00',
            'align': 'right'
        })
        
        # Format for percentage columns
        percent_format = workbook.add_format({
            'num_format': '0.00%',
            'align': 'right'
        })
        
        # Apply currency formatting to specific columns
        worksheet.set_column('F:H', 12, currency_format)  # Price Paid, Total Cost, Market Value
        worksheet.set_column('I:I', 12, currency_format)  # Total Gain
        
        # Apply percentage formatting
        worksheet.set_column('J:K', 12, percent_format)  # Total Gain %, Percent of Portfolio
        
        # Auto-adjust column widths
        for i, col in enumerate(consolidated_df.columns):
            column_len = max(consolidated_df[col].astype(str).map(len).max(), len(col))
            worksheet.set_column(i, i, column_len + 2)



In [ ]:
export_to_excel(consolidated_final)